# 1kg_eur - rare-variant PCA: frequency fractionation and loadings-vs-metadata

Two questions about the strange loadings, both on the existing panel, both local.

**Part A - where in the frequency range does it start?** Split the pruned rare
variants into equal-count MAF bins and refit the PCA within each. If the
phenomenon appears only in the lowest bin it is a low-count artefact; if it is
flat across bins it is structural; if it strengthens monotonically as MAF falls
that points at genotyping error, which scales the same way.

**Part B - does any variant-level feature predict the loadings?** Build a
per-variant feature table and regress squared loading on it. A feature that
"lights up" names the mechanism.

### What is NOT available, and why

**No singletons or doubletons.** `unified_panel_v9` was built with a pooled
`--maf 0.001` floor, and AoU's ACAF-threshold callset upstream only includes
variants passing its own AC/AF threshold - so singletons never reached the panel.
At N = 223,209 the lowest possible MAC is ~446. Screening them out is not
possible because they are already absent. The band available for fractionation is
0.1% <= MAF < 1%, i.e. MAC ~446-4,464 - a 10x range, enough to localise an onset.

**No per-variant genotype quality.** The ACAF pgen carries no GQ or DP. The
nearest proxies, all computable here:

| feature | why it might matter |
|---|---|
| `hwe_p`, `het_excess` | **01c skipped HWE filtering deliberately**, so error-prone sites are still in. Excess heterozygosity at a rare site is the classic signature of a mismapped or multi-mapping region |
| `f_miss`, `obs_ct` | differential missingness tracks assay trouble |
| `maf`, `log10_mac` | the frequency effect itself, as a covariate for the others |
| `is_indel`, `is_transition`, `is_cpg_like` | error-prone classes; CpG C>T is the canonical deamination artefact |
| `local_density` | variants per 100 kb - a proxy for repetitive or badly-mapped regions |
| `is_hm3` | separates the two arms' variant character |
| numeric `INFO` fields | discovered from the pvar header if present |

### Defining "the phenomenon"

Made explicit so the answer is not a matter of taste:

- `top1pct_share` - fraction of sum(w^2) in the top 1% of variants. Uniform = 0.01.
- `max_abs_z` - the most extreme individual on a PC, in SD units.
- `n_beyond_5sd` - individuals past 5 SD on any of PC1-5.

Adjust these if "strange loadings" means something else to you; everything
downstream reads from them.

**Reuses** the same panel and `.prune.in` files. No Batch, no downloads.

## Config

In [ ]:
import os, re, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_SET = "1kg_eur"
WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
OUT_GS = f"{WS_GS}/{SAMPLE_SET}/01_ancestry/rare_pca_fractionate"

RARE  = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare")
PANEL = f"{RARE}/qc/rare_qc"
PREV  = f"{RARE}/unrel"                 # rare_pca_unrelated.ipynb outputs
ARMS  = ["hm3_rare_all", "nonhm3_rare_all"]
KEEP  = f"{RARE}/round2_keep.txt"       # full sample; relatedness handling settled

OUT = f"{RARE}/fractionate"
os.makedirs(OUT, exist_ok=True)

N_PCS_FIT   = 10
N_BINS      = 4       # equal-VARIANT-COUNT MAF bins, so PCA noise is comparable
MIN_PER_BIN = 10_000  # an arm with fewer variants per bin gets fewer bins

for p in (f"{PANEL}.pgen", KEEP, *(f"{RARE}/{a}.prune.in" for a in ARMS)):
    print(f"  {'ok  ' if os.path.isfile(p) else 'MISS'} {p}")
print(f"\nout: {OUT}")

## What the pvar actually carries

Discovery, not assumption. Any numeric `INFO` key found here is added to the
Part B feature table automatically.

In [ ]:
hdr = subprocess.run(["bash", "-c", f"grep '^##' '{PANEL}.pvar' | head -40"],
                     capture_output=True, text=True).stdout
print(hdr or "(no ## header lines)")
colhdr = subprocess.run(["bash", "-c", f"grep -m1 '^#CHROM' '{PANEL}.pvar'"],
                        capture_output=True, text=True).stdout.strip()
print("columns:", colhdr)

INFO_KEYS = re.findall(r'##INFO=<ID=([^,]+)', hdr)
print("INFO keys:", INFO_KEYS or "(none)")
HAS_INFO = "INFO" in colhdr.split()
print("pvar has an INFO column:", HAS_INFO)

## Per-variant statistics

`--freq counts`, `--missing variant-only` and `--hardy midp`, computed once over
the union of both arms' pruned variants on the full sample. `midp` because the
exact HWE test is conservative at low counts.

In [ ]:
union_ids = f"{OUT}/union.ids"
ids = set()
for a in ARMS:
    ids |= {l.strip() for l in open(f"{RARE}/{a}.prune.in") if l.strip()}
with open(union_ids, "w") as fh:
    fh.write("\n".join(sorted(ids)) + "\n")
print(f"union of pruned variants: {len(ids):,}")

STATS = f"{OUT}/stats"
if not os.path.isfile(f"{STATS}.hardy"):
    subprocess.run(["bash", "-c", f"""
set -eo pipefail
export PATH="$HOME/bin:$PATH"
plink2 --pfile "{PANEL}" --nonfounders --keep "{KEEP}" --extract "{union_ids}" \
  --freq counts --missing variant-only --hardy midp \
  --threads $(nproc) --out "{STATS}"
"""], check=True)
else:
    print("stats exist, skipping")
subprocess.run(["bash", "-c", f"ls -la {OUT} | grep stats"], check=False)

In [ ]:
def pick(df, *names):
    """First matching column, tolerating plink2 header variants."""
    for n in names:
        if n in df.columns:
            return df[n]
    raise KeyError(f"none of {names} in {list(df.columns)}")

ac = pd.read_csv(f"{STATS}.acount", sep=r"\s+")
ac = ac.rename(columns={"#CHROM": "CHROM"})
alt = pick(ac, "ALT_CTS", "ALT_CT")
obs = pick(ac, "OBS_CT")
v = pd.DataFrame({
    "ID": ac["ID"], "CHROM": ac["CHROM"].astype(str).str.replace("^chr", "", regex=True),
    "REF": ac["REF"], "ALT": ac["ALT"],
    "mac": np.minimum(alt, obs - alt).astype(float),
    "obs_ct": obs.astype(float),
})
v["maf"] = v["mac"] / v["obs_ct"]

vm = pd.read_csv(f"{STATS}.vmiss", sep=r"\s+")
v = v.merge(vm[["ID", "F_MISS"]].rename(columns={"F_MISS": "f_miss"}), on="ID", how="left")

hw = pd.read_csv(f"{STATS}.hardy", sep=r"\s+")
ohet = pick(hw, "O(HET_A1)", "O(HET)")
ehet = pick(hw, "E(HET_A1)", "E(HET)")
hp   = pick(hw, "P", "MIDP")
v = v.merge(pd.DataFrame({"ID": hw["ID"], "hwe_p": hp.astype(float),
                          "het_excess": (ohet - ehet) / ehet.replace(0, np.nan)}),
            on="ID", how="left")

print(f"variants with stats: {len(v):,}")
print(v[["mac", "maf", "f_miss", "hwe_p", "het_excess"]].describe().round(5).to_string())
print(f"\nobserved MAC range: {v['mac'].min():.0f} - {v['mac'].max():.0f}")
print(f"observed MAF range: {v['maf'].min():.5f} - {v['maf'].max():.5f}")

# Part A - frequency fractionation

Equal-variant-count bins, so a difference between bins is not just a difference
in how many variants each PCA had to work with. Bin edges come from the observed
MAF distribution per arm, printed below.

In [ ]:
bins = {}
for arm in ARMS:
    arm_ids = {l.strip() for l in open(f"{RARE}/{arm}.prune.in") if l.strip()}
    a = v[v["ID"].isin(arm_ids)].sort_values("maf").reset_index(drop=True)
    nb = min(N_BINS, max(1, len(a) // MIN_PER_BIN))
    if nb < 2:
        print(f"{arm}: {len(a):,} variants - too few for {N_BINS} bins "
              f"(min {MIN_PER_BIN:,} each); skipping fractionation")
        continue
    edges = np.array_split(np.arange(len(a)), nb)
    for b, idx in enumerate(edges, 1):
        sub = a.iloc[idx]
        name = f"{arm}_b{b}"
        path = f"{OUT}/{name}.ids"
        sub["ID"].to_csv(path, index=False, header=False)
        bins[name] = {"arm": arm, "bin": b, "n": len(sub),
                      "maf_lo": sub["maf"].min(), "maf_hi": sub["maf"].max(),
                      "mac_lo": sub["mac"].min(), "mac_hi": sub["mac"].max(),
                      "ids": path}
    print(f"{arm}: {len(a):,} variants -> {nb} bins")

print()
print(pd.DataFrame(bins).T[["arm", "bin", "n", "maf_lo", "maf_hi", "mac_lo", "mac_hi"]]
      .to_string())

## Fit one PCA per bin

In [ ]:
for name, b in bins.items():
    stem = f"{OUT}/{name}"
    if os.path.isfile(f"{stem}.eigenvec"):
        print(f"exists, skipping: {name}")
        continue
    print(f"=== {name}  (n={b['n']:,}, MAF {b['maf_lo']:.4f}-{b['maf_hi']:.4f})")
    subprocess.run(["bash", "-c", f"""
set -eo pipefail
export PATH="$HOME/bin:$PATH"
plink2 --pfile "{PANEL}" --nonfounders --keep "{KEEP}" --extract "{b['ids']}" \
  --freq counts --pca approx {N_PCS_FIT} allele-wts \
  --threads $(nproc) --out "{stem}"
"""], check=True)

## Metrics per bin

The three definitions from the header, plus the full-arm fit as a reference row
so you can see whether a bin is worse or better than the arm as a whole.

In [ ]:
def metrics(stem):
    if not os.path.isfile(f"{stem}.eigenvec"):
        return None
    L = pd.read_csv(f"{stem}.eigenvec.allele", sep=r"\s+")
    E = pd.read_csv(f"{stem}.eigenvec", sep=r"\s+")
    ev = np.loadtxt(f"{stem}.eigenval")
    out = {"n_variants": len(L), "n_samples": len(E),
           "pc1_pct": 100 * ev[0] / ev.sum()}
    # loading concentration, PC1 and worst of PC1-5
    shares = []
    for k in range(1, 6):
        w2 = L[f"PC{k}"].to_numpy() ** 2
        n1 = max(1, len(w2) // 100)
        shares.append(np.sort(w2)[-n1:].sum() / w2.sum())
    out["top1pct_pc1"] = shares[0]
    out["top1pct_max_pc1to5"] = max(shares)
    # individual extremity
    z = E[[f"PC{k}" for k in range(1, 6)]].to_numpy()
    z = (z - z.mean(0)) / z.std(0)
    out["max_abs_z"] = np.abs(z).max()
    out["n_beyond_5sd"] = int((np.abs(z) > 5).any(axis=1).sum())
    return out

rows = []
for arm in ARMS:
    m = metrics(f"{PREV}/{arm}_all")
    if m:
        rows.append({"set": f"{arm} (full arm)", "arm": arm, "bin": 0,
                     "maf_lo": np.nan, "maf_hi": np.nan, **m})
for name, b in bins.items():
    m = metrics(f"{OUT}/{name}")
    if m:
        rows.append({"set": name, "arm": b["arm"], "bin": b["bin"],
                     "maf_lo": b["maf_lo"], "maf_hi": b["maf_hi"], **m})
frac = pd.DataFrame(rows)
print(frac.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, ttl in zip(
        axes,
        ["top1pct_max_pc1to5", "max_abs_z", "n_beyond_5sd"],
        ["loading concentration (top 1% share)", "most extreme individual (SD)",
         "individuals beyond 5 SD"]):
    for arm, c in zip(ARMS, ("tab:blue", "tab:orange")):
        d = frac[(frac["arm"] == arm) & (frac["bin"] > 0)].sort_values("maf_lo")
        if len(d):
            ax.plot(d["maf_lo"], d[col], "-o", color=c, label=arm)
        ref = frac[(frac["arm"] == arm) & (frac["bin"] == 0)]
        if len(ref):
            ax.axhline(ref[col].iloc[0], color=c, ls=":", lw=1,
                       label=f"{arm} full arm")
    ax.set_xscale("log"); ax.set_xlabel("bin lower MAF edge (log)")
    ax.set_title(ttl); ax.legend(fontsize=7)
if "top1pct_max_pc1to5" in frac:
    axes[0].axhline(0.01, color="0.5", ls="--", lw=1)
plt.suptitle("Where in the frequency range does the phenomenon appear?")
plt.tight_layout(); plt.savefig(f"{OUT}/fractionation.png", dpi=150); plt.show()

# Part B - loadings vs variant metadata

Outcome is squared loading, rescaled so the mean is 1 (so a coefficient reads as
"multiplies a variant's share of the PC"). Two views, because a linear model on a
heavy-tailed outcome is easy to mislead:

1. **Standardised OLS** of `w^2` on all features - coefficients comparable across
   features, plus each feature's partial R^2.
2. **AUC** for "is this variant in the top 1% by |loading|" - rank-based, immune
   to the tail, and directly interpretable as a screening statistic. 0.5 = no
   signal.

A feature with a large coefficient *and* an AUC away from 0.5 is a real lead.

In [ ]:
feat = v.copy()
feat["log10_mac"]     = np.log10(feat["mac"])
feat["neg_log10_hwe"] = -np.log10(feat["hwe_p"].clip(lower=1e-300))
feat["is_indel"]      = ((feat["REF"].str.len() > 1) | (feat["ALT"].str.len() > 1)).astype(float)
_ti = {("A","G"), ("G","A"), ("C","T"), ("T","C")}
feat["is_transition"] = [1.0 if (r, a) in _ti else 0.0
                         for r, a in zip(feat["REF"], feat["ALT"])]
# deamination proxy: C>T / G>A, the canonical artefact class
feat["is_cpg_like"]   = [1.0 if (r, a) in {("C","T"), ("G","A")} else 0.0
                         for r, a in zip(feat["REF"], feat["ALT"])]
# local density: pruned variants within +-100kb, from the ID position
feat["POS"] = pd.to_numeric(feat["ID"].str.split(":", n=2, expand=True)[1],
                            errors="coerce")
dens = []
for _c, g in feat.groupby("CHROM"):
    pos = np.sort(g["POS"].dropna().to_numpy())
    idx = g["POS"].to_numpy()
    lo = np.searchsorted(pos, idx - 100_000, "left")
    hi = np.searchsorted(pos, idx + 100_000, "right")
    dens.append(pd.Series(hi - lo, index=g.index))
feat["local_density"] = pd.concat(dens).reindex(feat.index).astype(float)

hm3_ids = {l.strip() for l in open(f"{RARE}/hm3_rare_all.prune.in") if l.strip()}
feat["is_hm3"] = feat["ID"].isin(hm3_ids).astype(float)

CANDIDATES = ["maf", "log10_mac", "obs_ct", "f_miss", "neg_log10_hwe", "het_excess",
              "is_indel", "is_transition", "is_cpg_like", "local_density", "is_hm3"]

# Coverage matters more than it looks. A feature with gaps would, under a plain
# dropna, silently redefine the analysis sample as "variants where that feature
# exists" - a selection nobody chose. So instead: median-impute the gaps, keep
# every variant, and add an explicit `<f>_isna` indicator so the missingness is
# itself a testable feature (plink2 not reporting HWE for a variant may well
# correlate with whatever is driving the loadings).
FEATURES, notes = [], []
for f in CANDIDATES:
    if f not in feat.columns or feat[f].notna().sum() == 0:
        notes.append(f"{f}: absent, dropped")
        continue
    cov = feat[f].notna().mean()
    if feat[f].std(skipna=True) == 0:
        notes.append(f"{f}: constant, dropped")
        continue
    if cov < 1.0:
        med = feat[f].median()
        feat[f + "_isna"] = feat[f].isna().astype(float)
        feat[f] = feat[f].fillna(med)
        FEATURES.append(f)
        if feat[f + "_isna"].std() > 0:
            FEATURES.append(f + "_isna")
        notes.append(f"{f}: {100*cov:.1f}% coverage - imputed to median "
                     f"{med:.4g}, added {f}_isna")
    else:
        FEATURES.append(f)

print("features:", FEATURES)
if notes:
    print("\ncoverage handling:")
    for n in notes:
        print("  " + n)
print()
print(feat[FEATURES].describe().round(4).to_string())

In [ ]:
def auc(score, label):
    """Mann-Whitney AUC, rank-based."""
    r = pd.Series(score).rank().to_numpy()
    n1 = label.sum(); n0 = len(label) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    return (r[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def analyse(stem, tag):
    if not os.path.isfile(f"{stem}.eigenvec.allele"):
        return None
    L = pd.read_csv(f"{stem}.eigenvec.allele", sep=r"\s+")
    idc = "#ID" if "#ID" in L.columns else "ID"
    d = L[[idc] + [f"PC{k}" for k in range(1, 6)]].rename(columns={idc: "ID"})
    n_before = len(d)
    d = d.merge(feat[["ID"] + FEATURES], on="ID", how="inner").dropna(subset=FEATURES)
    if len(d) < n_before:
        print(f"  {tag}: {n_before - len(d):,} of {n_before:,} variants lost at the "
              f"feature merge - check this, it should be 0 after imputation")
    X = d[FEATURES].to_numpy(float)
    X = (X - X.mean(0)) / X.std(0)
    Xd = np.column_stack([np.ones(len(X)), X])
    out = []
    for k in range(1, 6):
        w2 = d[f"PC{k}"].to_numpy(float) ** 2
        y = w2 / w2.mean()
        beta, *_ = np.linalg.lstsq(Xd, y, rcond=None)
        r2_full = 1 - ((y - Xd @ beta) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        top = y >= np.quantile(y, 0.99)
        for j, f in enumerate(FEATURES):
            keep = [c for c in range(Xd.shape[1]) if c != j + 1]
            b2, *_ = np.linalg.lstsq(Xd[:, keep], y, rcond=None)
            r2_wo = 1 - ((y - Xd[:, keep] @ b2) ** 2).sum() / ((y - y.mean()) ** 2).sum()
            out.append({"set": tag, "PC": k, "feature": f, "beta_std": beta[j + 1],
                        "partial_r2": r2_full - r2_wo, "auc_top1pct": auc(d[f], top),
                        "model_r2": r2_full, "n": len(d)})
    return pd.DataFrame(out)

res = pd.concat([r for r in (analyse(f"{PREV}/{a}_all", a) for a in ARMS)
                 if r is not None], ignore_index=True)
print(f"model R2 by arm and PC:")
print(res.groupby(["set", "PC"])["model_r2"].first().unstack().round(4).to_string())

In [ ]:
for tag in res["set"].unique():
    r = res[res["set"] == tag]
    print(f"\n{'=' * 70}\n{tag}  (n={int(r['n'].iloc[0]):,} variants)")
    for k in sorted(r["PC"].unique()):
        rk = (r[r["PC"] == k]
              .assign(strength=lambda x: x["partial_r2"].abs())
              .sort_values("strength", ascending=False))
        top = rk.head(4)
        line = ", ".join(
            f"{t.feature} (b={t.beta_std:+.3f}, pR2={t.partial_r2:.4f}, "
            f"AUC={t.auc_top1pct:.3f})" for t in top.itertuples())
        print(f"  PC{k} [R2={rk['model_r2'].iloc[0]:.4f}]: {line}")

In [ ]:
# heatmap of partial R2, features x PCs, per arm
for tag in res["set"].unique():
    piv = (res[res["set"] == tag]
           .pivot(index="feature", columns="PC", values="partial_r2")
           .reindex(FEATURES))
    fig, ax = plt.subplots(figsize=(7, 0.42 * len(FEATURES) + 2))
    m = np.abs(piv.to_numpy()).max() or 1
    im = ax.imshow(piv.to_numpy(), cmap="RdBu_r", vmin=-m, vmax=m, aspect="auto")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index, fontsize=8)
    ax.set_xlabel("PC"); ax.set_title(f"{tag} - partial R2 of each feature")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            val = piv.to_numpy()[i, j]
            if np.isfinite(val) and abs(val) > 0.005:
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.03)
    plt.tight_layout()
    plt.savefig(f"{OUT}/loadings_vs_metadata_{tag}.png", dpi=150); plt.show()

## Does the strongest feature survive conditioning on frequency?

The trap in Part B: nearly every quality feature correlates with MAF, so a
feature can look predictive purely as a stand-in for rarity. `maf` and
`log10_mac` are already in the model, so `partial_r2` is conditional on them -
but this repeats the ranking *within* the narrowest MAF bin, where frequency is
close to constant, as an independent check.

In [ ]:
narrow = {n: b for n, b in bins.items() if b["bin"] == 1}
res_n = []
for name, b in narrow.items():
    r = analyse(f"{OUT}/{name}", f"{name} (lowest MAF bin)")
    if r is not None:
        res_n.append(r)
if res_n:
    rn = pd.concat(res_n, ignore_index=True)
    for tag in rn["set"].unique():
        r = rn[rn["set"] == tag]
        print(f"\n{tag}")
        for k in sorted(r["PC"].unique())[:5]:
            rk = (r[r["PC"] == k].assign(s=lambda x: x["partial_r2"].abs())
                  .sort_values("s", ascending=False).head(3))
            print(f"  PC{k}: " + ", ".join(
                f"{t.feature} (pR2={t.partial_r2:.4f}, AUC={t.auc_top1pct:.3f})"
                for t in rk.itertuples()))
else:
    print("no binned fits available - run Part A first")

## Save and upload

In [ ]:
FRAC_TSV = f"{OUT}/fractionation_metrics.tsv"
META_TSV = f"{OUT}/loadings_vs_metadata.tsv"
frac.to_csv(FRAC_TSV, sep="\t", index=False, float_format="%.6f")
res.to_csv(META_TSV, sep="\t", index=False, float_format="%.6f")
print(f"wrote {FRAC_TSV}\nwrote {META_TSV}")

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{FRAC_TSV}" "{META_TSV}" "{OUT_GS}/"
gcloud storage cp "{OUT}/"*.png "{OUT_GS}/figures/"
gcloud storage cp "{OUT}/"*.eigenval "{OUT_GS}/bins/"
"""], check=True)
print(f"uploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_fractionate.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_fractionate.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')